In [1]:
# =============================================================================
# ABIDE I Data Download - GLOBAL SIGNAL REGRESSION (FIXED)
# For Kaggle Notebook
#
# This notebook downloads fALFF and ROI time series data
# preprocessed WITH global signal regression (filt_global strategy)
# for your specific 182 subjects
# =============================================================================

# CELL 1: Setup
# -----------------------------------------------------------------------------
import os
import urllib.request as request
import pandas as pd
import shutil

# Create output directories (GSR version)
os.makedirs('/kaggle/working/abide_global/falff', exist_ok=True)
os.makedirs('/kaggle/working/abide_global/rois_cc200', exist_ok=True)
print("Directories created!")

# =============================================================================
# CELL 2: Your 182 Subject IDs
# =============================================================================

SUBJECT_IDS = [
    50499, 50694, 51020, 51470, 50518, 51025, 50232, 50490, 50520, 50704,
    50245, 50514, 51467, 50530, 50236, 51585, 50521, 51016, 50241, 51468,
    50134, 51021, 51459, 50702, 50024, 51026, 50012, 50482, 50689, 50004,
    51023, 50711, 51318, 50486, 51329, 50237, 50130, 50399, 51027, 51322,
    50485, 50254, 50121, 50496, 51029, 51017, 51328, 51571, 51474, 50320,
    50690, 51466, 50973, 51572, 50693, 50251, 50498, 50487, 50500, 50233,
    50136, 51473, 50696, 50246, 50700, 51015, 50695, 50131, 50686, 51237,
    51457, 51464, 50507, 51607, 50488, 51018, 50516, 50649, 50480, 50491,
    50504, 50020, 50253, 51019, 50503, 50184, 50007, 51028, 50527, 51465,
    50415, 50388, 50442, 50267, 51077, 50382, 51147, 51067, 50440, 51559,
    50446, 50345, 50469, 51150, 50339, 50107, 50044, 51365, 51113, 51268,
    51146, 50449, 50444, 50030, 51118, 50032, 50445, 50115, 50368, 50060,
    50703, 51344, 51151, 51114, 50259, 51334, 51131, 50035, 50710, 51558,
    51132, 50058, 50270, 51152, 51148, 50260, 50261, 51560, 50344, 50037,
    51153, 51112, 50385, 51332, 50706, 50439, 51557, 50329, 50391, 51139,
    51346, 51117, 50041, 51364, 50040, 50349, 51156, 50046, 51130, 50117,
    50709, 51339, 50441, 51149, 51116, 51347, 50263, 51340, 50335, 51129,
    51335, 50386, 51333, 50443, 50434, 50424, 50707, 50467, 50433, 50463,
    51342, 51066
]

print(f"Total subjects: {len(SUBJECT_IDS)}")

# =============================================================================
# CELL 3: Load ABIDE Phenotype File
# =============================================================================

S3_PREFIX = 'https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative'
PHENO_URL = f'{S3_PREFIX}/Phenotypic_V1_0b_preprocessed1.csv'

print("Downloading phenotype file...")
pheno_df = pd.read_csv(PHENO_URL)
print(f"Phenotype file loaded: {len(pheno_df)} subjects")

sub_to_file = dict(zip(pheno_df['SUB_ID'], pheno_df['FILE_ID']))

found = sum(1 for s in SUBJECT_IDS if s in sub_to_file and sub_to_file[s] != 'no_filename')
print(f"Found {found}/{len(SUBJECT_IDS)} subjects with valid FILE_IDs")

# =============================================================================
# CELL 4: Download Function (FORCED GSR ONLY)
# =============================================================================

GSR_STRATEGY = 'filt_global'

def download_file(file_id, derivative, pipeline='cpac'):
    """Download ONLY GSR-preprocessed ABIDE data."""
    
    if 'roi' in derivative.lower():
        extension = '.1D'
        out_dir = '/kaggle/working/abide_global/rois_cc200'
    else:
        extension = '.nii.gz'
        out_dir = '/kaggle/working/abide_global/falff'

    filename = f"{file_id}_{derivative}{extension}"

    s3_url = (
        f"{S3_PREFIX}/Outputs/"
        f"{pipeline}/{GSR_STRATEGY}/{derivative}/{filename}"
    )

    # Safety check (prevents accidental non-GSR download)
    assert '/filt_global/' in s3_url, f"Non-GSR path blocked: {s3_url}"

    local_path = os.path.join(out_dir, filename)

    if os.path.exists(local_path):
        return True, "exists"

    try:
        request.urlretrieve(s3_url, local_path)
        return True, "downloaded"
    except Exception as e:
        return False, str(e)

# =============================================================================
# CELL 5: Download fALFF (GSR)
# =============================================================================

print("\n" + "="*60)
print("DOWNLOADING fALFF - GLOBAL SIGNAL REGRESSION")
print(f"Strategy: {GSR_STRATEGY}")
print("="*60 + "\n")

stats = {'downloaded': 0, 'exists': 0, 'failed': 0, 'missing': 0}

for i, sub_id in enumerate(SUBJECT_IDS):
    if sub_id not in sub_to_file or sub_to_file[sub_id] == 'no_filename':
        stats['missing'] += 1
        continue

    file_id = sub_to_file[sub_id]
    success, status = download_file(file_id, 'falff')

    if success:
        stats[status] += 1
    else:
        stats['failed'] += 1
        print(f"FAILED [{i+1}]: {file_id} - {status}")

    if (i+1) % 30 == 0:
        print(f"Progress: {i+1}/{len(SUBJECT_IDS)} | Downloaded: {stats['downloaded']} | Exists: {stats['exists']}")

print(f"\nfALFF Complete: {stats}")

# =============================================================================
# CELL 6: Download ROIs CC200 (GSR)
# =============================================================================

print("\n" + "="*60)
print("DOWNLOADING ROIs CC200 - GLOBAL SIGNAL REGRESSION")
print(f"Strategy: {GSR_STRATEGY}")
print("="*60 + "\n")

stats = {'downloaded': 0, 'exists': 0, 'failed': 0, 'missing': 0}

for i, sub_id in enumerate(SUBJECT_IDS):
    if sub_id not in sub_to_file or sub_to_file[sub_id] == 'no_filename':
        stats['missing'] += 1
        continue

    file_id = sub_to_file[sub_id]
    success, status = download_file(file_id, 'rois_cc200')

    if success:
        stats[status] += 1
    else:
        stats['failed'] += 1
        print(f"FAILED [{i+1}]: {file_id} - {status}")

    if (i+1) % 30 == 0:
        print(f"Progress: {i+1}/{len(SUBJECT_IDS)} | Downloaded: {stats['downloaded']} | Exists: {stats['exists']}")

print(f"\nROIs Complete: {stats}")

# =============================================================================
# CELL 7: Verification
# =============================================================================

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

falff_dir = '/kaggle/working/abide_global/falff'
rois_dir = '/kaggle/working/abide_global/rois_cc200'

falff_files = [f for f in os.listdir(falff_dir) if f.endswith('.nii.gz')]
rois_files = [f for f in os.listdir(rois_dir) if f.endswith('.1D')]

print(f"\nfALFF files: {len(falff_files)}")
print(f"ROIs CC200 files: {len(rois_files)}")

# =============================================================================
# CELL 8: ZIP OUTPUT
# =============================================================================

zip_path = '/kaggle/working/abide_global_filt_global'
shutil.make_archive(zip_path, 'zip', '/kaggle/working/abide_global')

print(f"\nCreated ZIP: {zip_path}.zip")

# =============================================================================
# CELL 9: Mapping File
# =============================================================================

mapping_data = []

for sub_id in SUBJECT_IDS:
    if sub_id in sub_to_file and sub_to_file[sub_id] != 'no_filename':
        file_id = sub_to_file[sub_id]

        falff_file = f"{file_id}_falff.nii.gz"
        rois_file = f"{file_id}_rois_cc200.1D"

        mapping_data.append({
            'SUB_ID': sub_id,
            'FILE_ID': file_id,
            'falff_file': falff_file,
            'rois_file': rois_file
        })

mapping_df = pd.DataFrame(mapping_data)
mapping_df.to_csv('/kaggle/working/subject_file_mapping.csv', index=False)

print("\nDONE!")
print(mapping_df.head())

Directories created!
Total subjects: 182
Phenotype file loaded: 1112 subjects
Found 182/182 subjects with valid FILE_IDs

DOWNLOADING fALFF - GLOBAL SIGNAL REGRESSION
Strategy: filt_global

Progress: 30/182 | Downloaded: 30 | Exists: 0
Progress: 60/182 | Downloaded: 60 | Exists: 0
Progress: 90/182 | Downloaded: 90 | Exists: 0
Progress: 120/182 | Downloaded: 120 | Exists: 0
Progress: 150/182 | Downloaded: 150 | Exists: 0
Progress: 180/182 | Downloaded: 180 | Exists: 0

fALFF Complete: {'downloaded': 182, 'exists': 0, 'failed': 0, 'missing': 0}

DOWNLOADING ROIs CC200 - GLOBAL SIGNAL REGRESSION
Strategy: filt_global

Progress: 30/182 | Downloaded: 30 | Exists: 0
Progress: 60/182 | Downloaded: 60 | Exists: 0
Progress: 90/182 | Downloaded: 90 | Exists: 0
Progress: 120/182 | Downloaded: 120 | Exists: 0
Progress: 150/182 | Downloaded: 150 | Exists: 0
Progress: 180/182 | Downloaded: 180 | Exists: 0

ROIs Complete: {'downloaded': 182, 'exists': 0, 'failed': 0, 'missing': 0}

VERIFICATION

fALF